In [5]:
import os
import requests
import csv  # Importamos el módulo csv para manejar archivos CSV correctamente
import re

In [ ]:
    """
    Descarga imágenes desde URLs especificadas en un archivo CSV y las guarda
    en subdirectorios basados en la primera parte del nombre del archivo (la categoría).

    Args:
        csv_filepath (str): La ruta al archivo CSV.
        base_output_directory (str): El nombre del directorio base donde se crearán los subdirectorios.
    """    """
    Descarga imágenes desde URLs especificadas en un archivo CSV y las guarda
    en subdirectorios basados en la primera parte del nombre del archivo (la categoría).

    Args:
        csv_filepath (str): La ruta al archivo CSV.
        base_output_directory (str): El nombre del directorio base donde se crearán los subdirectorios.
    """

In [8]:
def descargar_imagenes_organizadas_desde_csv(csv_filepath, base_output_directory="dataset_organizado"):
    if not os.path.exists(base_output_directory):
      os.makedirs(base_output_directory)
      print(f"Directorio base '{base_output_directory}' creado.")
    else:
      print(f"El directorio base '{base_output_directory}' ya existe.")

    try:
      with open(csv_filepath, 'r', newline='', encoding='utf-8') as f:
        # Usamos csv.reader para manejar correctamente las comas y las comillas
        csv_reader = csv.reader(f)
        for line_num, row in enumerate(csv_reader, 1):
          # row será una lista: ['ammonites', '00000.png', 'https://...']
          if len(row) < 3:
            print(
                f"Advertencia: Línea {line_num} ignorada por formato incorrecto (menos de 3 campos): '{','.join(row)}'"
            )
            continue

          category_part = row[0]  # Ej: "ammonites"
          filename_part = row[1]  # Ej: "00000.png"
          image_url = row[2]  # Ej: "https://www.steinkern.de/images/berichte/..."

          full_filename = f"{category_part}{filename_part}"  # Reconstruimos el nombre completo: ammonites00000.png

          # La categoría ya la tenemos del primer campo del CSV
          category = category_part

          # Crear el directorio para la categoría si no existe
          category_output_path = os.path.join(base_output_directory, category)
          if not os.path.exists(category_output_path):
            os.makedirs(category_output_path)
            print(f"Subdirectorio '{category_output_path}' creado.")

          filepath = os.path.join(category_output_path, full_filename)

          try:
            print(
                f"Descargando {full_filename} en la categoría '{category}' desde {image_url}..."
            )
            response = requests.get(image_url, stream=True,
                                    timeout=10)  # Añadir timeout para evitar esperas infinitas
            response.raise_for_status(
            )  # Lanza un error para códigos de estado HTTP malos (4xx o 5xx)

            with open(filepath, 'wb') as img_file:
              for chunk in response.iter_content(1024):
                img_file.write(chunk)
            print(f"Guardado: {filepath}")

          except requests.exceptions.RequestException as e:
            print(f"Error al descargar {full_filename} desde {image_url}: {e}")
          except IOError as e:
            print(f"Error al guardar {filepath}: {e}")
          except Exception as e:
            print(f"Ocurrió un error inesperado con {full_filename}: {e}")

    except FileNotFoundError:
      print(f"Error: El archivo CSV '{csv_filepath}' no fue encontrado.")
    except Exception as e:
      print(f"Error al leer el archivo CSV: {e}")


In [9]:
 # --- USO DEL SCRIPT ---
if __name__ == "__main__":
    csv_file = "fossilnet-image-uri.csv" # ¡Asegúrate de cambiar esto por el nombre real de tu archivo CSV!
    descargar_imagenes_organizadas_desde_csv(csv_file)
    print("\nProceso de descarga y organización completado.")

Directorio base 'dataset_organizado' creado.
Subdirectorio 'dataset_organizado\ammonites' creado.
Descargando ammonites00000.png en la categoría 'ammonites' desde https://www.steinkern.de/images/berichte/Gastbeitraege5/0_ur2/35_1.JPG...
Guardado: dataset_organizado\ammonites\ammonites00000.png
Descargando ammonites00001.png en la categoría 'ammonites' desde https://api.europeana.eu/api/v2/thumbnail-by-url.json?uri=http%3A%2F%2Fwww.museum-digital.de%2Fdata%2Fwestfalen%2Fimages%2F201309%2F08163958208.jpg&type=IMAGE...
Guardado: dataset_organizado\ammonites\ammonites00001.png
Descargando ammonites00002.png en la categoría 'ammonites' desde https://lh5.googleusercontent.com/proxy/2eXSCycynHaM8-BirMlT-9JeCHrDxKMgucWddmyFlVv5_eFc4h6r2Q82byxfFzoxPKoVdWX58TN3PgHbAl6hztfs9wwSPOAbSvXTSjs6x3eEtgAfARsH6Z4oKw...
Guardado: dataset_organizado\ammonites\ammonites00002.png
Descargando ammonites00003.png en la categoría 'ammonites' desde https://a4.pbase.com/t9/41/984841/4/162204302.qlJ8IsbW.jpg...
Guar